# Figure 3. Glycoproteome-wide association analysis for Alzheimer's disease

In [ ]:
library(ggplot2)
library(ggrepel)
library(dplyr)

# Add a column for MR significance
df <- readRDS('data//Figure_3a.rds')$df
unadj_cutoff <- readRDS('data//Figure_3a.rds')$unadj_cutoff
axis_df <- readRDS('data//Figure_3a.rds')$axis_df

label_mode <- "gene"   # "gene": one label per gene (deduplicated); set to "peptiform" to label glycopeptiforms
# label every gene with a GlycoWAS-significant glycopeptiform
lab_df <- df %>% filter(pass)
lab_df <- if (label_mode == "gene") {
  lab_df %>% group_by(gene_name) %>% slice_max(abs(logp), n = 1, with_ties = FALSE) %>% ungroup() %>% mutate(.lab = gene_name)
} else {
  lab_df %>% mutate(.lab = peptide)
}

df <- df %>%
  mutate(
    chr_num = as.integer(as.character(chr)),
    point_group = case_when(
      is_coloc & pass ~ "Yes",   # colocalized points without GlycoWAS significance stay grey
      chr_num %% 2 == 1 ~ "No (odd)",
      TRUE ~ "No (even)"
    )
  ) %>% arrange(point_group == "Yes")

f3a <- ggplot(df, aes(x = cum_pos, y = logp)) +
  geom_point(aes(color = point_group, shape = MR), size = 2, alpha = 0.8) +
  scale_color_manual(
    name = "ADRD colocalization",
    values = c("Yes" = "red", "No (odd)" = "grey80", "No (even)" = "#C6DBEF"),
    breaks = c("Yes", "No (odd)"),
    labels = c("Yes", "No"),
    guide = guide_legend(order = 2)
  ) +
  scale_shape_manual(
    name = "MR",
    values = c("Significant" = 17, "Not Significant" = 16), # 17 = triangle, 16 = circle
    labels = c("Not Significant" = "Not significant"),
    guide = guide_legend(order = 1)   # MR legend leftmost
  ) +
  geom_hline(yintercept = 0, linetype = "dashed") +
  geom_hline(
    yintercept = c(-log10(unadj_cutoff), log10(unadj_cutoff)), 
    linetype = "dashed", color = "red"
  ) +
  geom_text_repel(
    data = lab_df,
    aes(label = .lab),
    size = 4,
    color = "steelblue",
    box.padding = 0.3,
    max.overlaps = 50
  ) +
  scale_x_continuous(
    breaks = axis_df$center,
    labels = axis_df$chr
  ) +
  labs(
    x = "Chromosome",
    y = expression(paste("GlycoWAS ", -log[10](P)))
  ) +
  theme_classic(base_size = 16) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    panel.grid = element_blank(),
    axis.line = element_line(color = "black", linewidth = 0.35),
    axis.ticks = element_line(color = "black", linewidth = 0.35),
    legend.position = "bottom"
  )

## Figure 3B right — two bars over AD-colocalized glycopeptiforms

In [ ]:
library(data.table); library(ggplot2); library(patchwork)
b <- readRDS('data/Figure_3b_bars.rds')
dir_twas <- b$dir_twas; gene_order_twas <- b$gene_order_twas
all_gp_dt <- copy(b$all_gp_dt)
glycan_class_colors <- b$glycan_class_colors
lvl <- names(glycan_class_colors)                       # common glycan-class order for a shared legend
glycan_class_labels <- c("Hybrid/Complex" = "Hybrid/complex")  # sentence case for legend display only
int_breaks <- function(x) { br <- pretty(x); br[abs(br - round(br)) < 1e-9] }
bar_thm <- theme_minimal(base_size = 14) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1), axis.text = element_text(size = 12),
        panel.grid = element_blank(), axis.line = element_line(linewidth = 0.3, color = "grey30"))

# Panel b: ALL glycopeptiforms
all_gp_dt[, GlycanClass := factor(GlycanClass, levels = lvl)]
f3b_all <- ggplot(all_gp_dt, aes(factor(gene_name, levels = gene_order_twas), N, fill = GlycanClass)) +
  geom_col(width = 0.7) +
  scale_fill_manual(values = glycan_class_colors, name = "Glycan class", drop = FALSE, labels = glycan_class_labels) +
  scale_y_continuous(breaks = int_breaks) +
  labs(x = NULL, y = "Glycopeptiforms") + bar_thm +
  theme(plot.margin = margin(t = 10, r = 35, b = 10, l = 35),
        plot.tag.position = c(-0.05, 1.0))

coloc_cnt <- dir_twas[, .N, by = .(gene_name, GlycanClass, direction)]
coloc_cnt[, GlycanClass := factor(GlycanClass, levels = lvl)]
coloc_cnt[, y := ifelse(direction == "Negative", -N, N)]
f3b2_count <- ggplot(coloc_cnt, aes(factor(gene_name, levels = gene_order_twas), y, fill = GlycanClass)) +
  geom_col(width = 0.7) +
  geom_hline(yintercept = 0, color = "grey40", linewidth = 0.4) +
  scale_fill_manual(values = glycan_class_colors, name = "Glycan class", drop = FALSE, labels = glycan_class_labels) +
  scale_y_continuous(breaks = int_breaks, labels = function(x) abs(x)) +
  labs(x = NULL, y = "ADRD GWAS-colocalized\nglycopeptiforms") + bar_thm +
  guides(fill = "none") +   # shared legend lives on panel b, which has real data for every class
  theme(plot.margin = margin(t = 10, r = 5, b = 10, l = 65))

options(repr.plot.width = 12, repr.plot.height = 6)
(f3b_all | f3b2_count) + plot_layout(guides = "collect")

## Combine

In [ ]:
library(patchwork); library(cowplot)
# b = all glycopeptiforms by glycan class
bars <- ((f3b_all + labs(tag = "b")) | (f3b2_count + labs(tag = "c"))) +
  plot_layout(guides = "collect") &
  theme(plot.tag = element_text(face = "bold", size = 22))
bottom_row <- bars   # pass patchwork directly to plot_grid (a lone wrap_elements renders blank)

options(repr.plot.width = 18, repr.plot.height = 10)
top_row <- f3a
combined_fig <- plot_grid(
  top_row, bottom_row, ncol = 1,
  rel_heights = c(2, 1.3),
  labels = c("a", ""),
  label_fontface = "bold", label_size = 22, label_x = 0, label_y = 1
)
combined_fig

ggsave("Figure3.pdf", combined_fig, width = 14, height = 10, dpi = 300)